# MSBA 6331 Homework 1 - Big Data and Hive

This assignment is based on the introduction to Big data module and the Introduction to Hive module. Please complete the lectures and labs in this modules before this assignment. 

Read and review the [general homework guidelines](https://de4liu.github.io/bigdata/FAQs/homework-guidelines/). 

All homework assignments are individual assignments. While you may still discuss homework problems with fellow students, no documents or fragments of homework paper should be shared between students, physically or digitally.

Always properly cite Internet and other sources if you use any.  


## Part I. Short Answers
1\. **Share Nothing**: Explain what's "share nothing" architecture, how does it contributes to handling big data? 

> If you use GenAI for this question, make sure you understand what it is saying by asking follow up questions and fact/quality the generated statements as you will be responsible for them. 

Ans 1: 
Shared-nothing architecture allows parallel tasks to run 
independently without sharing memory or storage. 

There is no central memory or storage, therefore, tasks do not compete for shared resources. Each machine processes its 
own portion of data using its own 
CPU, RAM, and disk. This enables massive parallelism, which is essential when working with very large datasets.

This architecture is particularly well suited for MapReduce, where the Map phase processes data in parallel across multiple machines and the Reduce phase aggregates the results. Because MapReduce assigns independent tasks to different nodes, shared-nothing architecture allows these tasks to execute efficiently without resource contention.

As data volume grows, the system can scale horizontally by adding more machines instead of upgrading a 
single server. This makes it possible to handle the large datasets, that is, the terabytes or petabytes of data efficiently.

Additionally, because tasks execute independently, failures are isolated to individual machines. 
The system can recover by reassigning tasks, making Big Data processing more reliable and fault tolerant.

2. **Managed and External Tables**:  In your own words, what are the main differences between Managed tables and External tables in Hive? Give an example when you should a Managed table and when you should use an external table.

Ans 2:
Managed tables:

1. Hive owns and manages both data and metadata
2. Data is stored by default in /user/hive/warehouse in HDFS diretory
3. Each managed table has a subdirectory named after the table
4. A table’s directory may contain multiple data files
5. When a managed table is dropped, both metadata and data files are deleted
6. It is used when Hive is responsible for the entire data lifecycle
For example, a company can create a managed table to store temporary sales summaries or intermediate 
results which are generated during data processing. This data would be used only within Hive and can be safely 
removed when no longer needed. Also, dropping the table can also  be used to delete the data.

External Tables:
1. Hive manages only the metadata, not the actual data
2. Data can be stored in a user-defined location
3. Data may reside elsewhere in HDFS or in cloud storage
4. Dropping an external table removes only the table definition 
5. Data files remain unchanged
6. It is used when data is shared across systems or must be preserved
For example, raw website log files or customer data that is stored in HDFS or cloud storage and is to be accessed by tools 
such as Hive and Spark. In this case, Hive should only reference the data for querying, and dropping the table 
should not delete the underlying files.


## Part II.  Hands-On 

This part of the assignment uses the Databricks Free Edition. 

### A. Linux commands

`access.log` contains web access logs with information such as visitor's ip address, time/data, the page requested, the http protocol, HTTP status code, bytes transmitted, referral, visitor's browser, and sessionid.

Use the **Linux commands** to do the following:

1. Verify your working directory

In [ ]:
pwd

2. Download and unzip [adir.zip](https://idsdl.csom.umn.edu/c/share/msba6330/adir.zip) to the Serverless instance (which is a Linux instance) attached to this notebook.



In [ ]:
!ls




 access.log		        latlon.tsv
 ad_data1.txt		        ratings_2012.txt
 ad_data2.txt		        ratings_2013.txt
 Drafts			       'Untitled File 2026-01-31 11_57_34.py'
 homework1.ipynb	       'Untitled File 2026-01-31 12_00_42.py'
 hw1			       'Untitled Notebook 2026-02-08 00:47:45.ipynb'
 lab3-1-hiveIntro_Parul.ipynb


3. Find out the number of lines in `access.log`

In [ ]:
!!wc -l access.log


In [ ]:

!wc -l access.log


412860 access.log


4. Display the first 20 lines of the file. (tip: use `-20` as command option for `head`)

As a background, each line in the access log records **one request to the website**.

* **IP address** – where the visitor’s request came from
* **Date & time** – when the request happened
* **Request** – what page or file was requested
* **Status code** – whether the request succeeded (e.g., 200 = OK, 500 = error)
* **Response size** – how much data was sent back
* **Referrer** – where the visitor came from (e.g., Google, another site)
* **Browser info** – the visitor’s browser and operating system
* **Session ID** – an identifier linking multiple requests from the same visit



In [ ]:
!head -20 access.log


10.34.198.119 - - [31/May/2013:00:00:00 -0800] "GET /about/careers HTTP/1.1" 200 48091 "http://www.google.com/search?q=laptop" "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.31 (KHTML, like Gecko) Chrome/26.0.1410.43 Safari/537.31" "SESSIONID=743021225616"
192.168.250.171 - - [31/May/2013:00:00:02 -0800] "GET /contact HTTP/1.1" 500 176 "http://www.example.com/" "Mozilla/5.0 (Windows NT 6.0; rv:19.0) Gecko/20100101 Firefox/19.0" "SESSIONID=909881341477"
192.168.250.171 - - [31/May/2013:00:00:02 -0800] "GET /css/style.css HTTP/1.1" 304 0 "http://www.example.com/" "Mozilla/5.0 (Windows NT 6.0; rv:19.0) Gecko/20100101 Firefox/19.0" "SESSIONID=909881341477"
192.168.250.171 - - [31/May/2013:00:00:02 -0800] "GET /img/beacon.png HTTP/1.1" 304 0 "http://www.example.com/" "Mozilla/5.0 (Windows NT 6.0; rv:19.0) Gecko/20100101 Firefox/19.0" "SESSIONID=909881341477"
192.168.250.171 - - [31/May/2013:00:00:02 -0800] "GET /img/dualcore-logo.png HTTP/1.1" 200 5692 "http://www.example.com/" "Mozil

5. Create a subfolder `hw1` in your current directory

In [ ]:
!mkdir hw1
!ls

mkdir: cannot create directory ‘hw1’: File exists
 access.log		        latlon.tsv
 ad_data1.txt		        ratings_2012.txt
 ad_data2.txt		        ratings_2013.txt
 Drafts			       'Untitled File 2026-01-31 11_57_34.py'
 homework1.ipynb	       'Untitled File 2026-01-31 12_00_42.py'
 hw1			       'Untitled Notebook 2026-02-08 00:47:45.ipynb'
 lab3-1-hiveIntro_Parul.ipynb


6. Take the last 1000 records of `access.log` and save them as `access_sample.log` in the newly created directory.

In [ ]:
!tail -1000 access.log > hw1/access_sample.log
!ls hw1

access_sample.log


7. Count the number of lines in `hw1/access_sample.log`

In [ ]:
!wc -l hw1/access_sample.log


1000 hw1/access_sample.log


### B. Hive - Import and Query Flights Data




#### Download data files, upload to Databricks, and inspect them.

1. Download the [airport_data.zip](https://idsdl.csom.umn.edu/c/share/msba6330/airport_data.zip) from the web.
2. Extract the files from the zip file to a local folder
3. Upload the data to Databricks Catalog  /workspace/msbabigdata/flights (tip: if you have gone through Lab 3-1, the workspace msbabigdata has already been created, you just need to create a new **managed volume** `flights` in the existing schema `msbabigdata`). This workspace is for saving our input data in files before they become a table


4\. Verify the content of the target volume using Databricks File System command `%fs ls /Volumes/workspace/msbabigdata/flights/` (there should be 12 csv files)

> Tip: Databricks has its own file system for managing files you upload in the workspace. It is not Linux file system but it supports `ls` and `head` (without much options). We are using these to list files and see what are in these files. The `%fs` prefix is indicating to databricks these are Databrick file system commands.


In [ ]:
%fs ls /Volumes/workspace/msbabigdata/flights/

path | name | size | modificationTime
dbfs:/Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_1.csv | 297004386_T_ONTIME_1.csv | 54599300 | 1770613568000
dbfs:/Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_2.csv | 297004386_T_ONTIME_2.csv | 51994206 | 1770613567000
dbfs:/Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_3.csv | 297004386_T_ONTIME_3.csv | 58775991 | 1770613570000
dbfs:/Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_4.csv | 297004386_T_ONTIME_4.csv | 56643780 | 1770613569000
dbfs:/Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_5.csv | 297004386_T_ONTIME_5.csv | 58812061 | 1770613570000
dbfs:/Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_6.csv | 297004386_T_ONTIME_6.csv | 59791503 | 1770613570000
dbfs:/Volumes/workspace/msbabigdata/flights/297043733_T_ONTIME_10.csv | 297043733_T_ONTIME_10.csv | 58450839 | 1770613569000
dbfs:/Volumes/workspace/msbabigdata/flights/297043733_T_ONTIME_11.csv | 297043733_T_ONTIME_11.csv |

5. Show the content of one of the files using Databricks file system's command `%fs head /Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_1.csv`. This helps you to decide later on how to ingest these data files.

> Unlike Linux's `head` which shows the first 10 rows by default, `%fs head` shows the first 64 kilobytes. 

In [ ]:
%fs head /Volumes/workspace/msbabigdata/flights/297004386_T_ONTIME_1.csv

[Truncated to first 65536 bytes]
"YEAR","MONTH","DAY_OF_MONTH","FL_DATE","UNIQUE_CARRIER","AIRLINE_ID","CARRIER","TAIL_NUM","FL_NUM","ORIGIN_AIRPORT_ID","ORIGIN_AIRPORT_SEQ_ID","ORIGIN","DEST_AIRPORT_ID","DEST_AIRPORT_SEQ_ID","DEST","DEP_DELAY","ARR_DELAY","CANCELLED","DIVERTED","DISTANCE",
2016,1,6,2016-01-06,"AA",19805,"AA","N4YBAA","43",11298,1129804,"DFW",11433,1143302,"DTW",-3.00,-6.00,0.00,0.00,986.00,
2016,1,7,2016-01-07,"AA",19805,"AA","N434AA","43",11298,1129804,"DFW",11433,1143302,"DTW",-4.00,-12.00,0.00,0.00,986.00,
2016,1,8,2016-01-08,"AA",19805,"AA","N541AA","43",11298,1129804,"DFW",11433,1143302,"DTW",-5.00,7.00,0.00,0.00,986.00,
2016,1,9,2016-01-09,"AA",19805,"AA","N489AA","43",11298,1129804,"DFW",11433,1143302,"DTW",2.00,-5.00,0.00,0.00,986.00,
2016,1,10,2016-01-10,"AA",19805,"AA","N439AA","43",11298,1129804,"DFW",11433,1143302,"DTW",100.00,113.00,0.00,0.00,986.00,
2016,1,11,2016-01-11,"AA",19805,"AA","N468AA","43",11298,1129804,"DFW",11433,1143302,"DTW",7.00,-3.00,0.00

#### Directly Query the csv files

On Databricks, it is possible to directly query structured data sources such as CSV, JSON, XML etc without creating a table first. This is done using the SQL function `read_files`:

For example, in the following, we read files from a Unity Catalog folder, specifying the format of the files to be `csv`, have a header row, and provided an explicit schema for the csv file. 


```sql
read_files("/Volumes/workspace/msbabigdata/flights/",
  format => "csv",
  schema => "YEAR STRING, MONTH STRING, DAY_OF_MONTH STRING, FL_DATE STRING, UNIQUE_CARRIER STRING, AIRLINE_ID STRING, CARRIER STRING, TAIL_NUM STRING, FL_NUM STRING, ORIGIN_AIRPORT_ID STRING, ORIGIN_AIRPORT_SEQ_ID STRING, ORIGIN STRING, DEST_AIRPORT_ID STRING, DEST_AIRPORT_SEQ_ID STRING, DEST STRING, DEP_DELAY STRING, ARR_DELAY STRING, CANCELLED STRING, DIVERTED STRING, DISTANCE STRING",
  header => "true"
)
```

You can use this function to as if it is a table. e.g. `select * from read_files(....)`


6. Directly query the uploaded csv files and show the first 10 rows.

We use this to ensure that we load the CSV file correctly. You should make sure that the results have the right field names, types, and values

In [ ]:
%sql
SELECT *
FROM read_files(
  "/Volumes/workspace/msbabigdata/flights/",
  format => "csv",
  schema => "
    YEAR STRING,
    MONTH STRING,
    DAY_OF_MONTH STRING,
    FL_DATE STRING,
    UNIQUE_CARRIER STRING,
    AIRLINE_ID STRING,
    CARRIER STRING,
    TAIL_NUM STRING,
    FL_NUM STRING,
    ORIGIN_AIRPORT_ID STRING,
    ORIGIN_AIRPORT_SEQ_ID STRING,
    ORIGIN STRING,
    DEST_AIRPORT_ID STRING,
    DEST_AIRPORT_SEQ_ID STRING,
    DEST STRING,
    DEP_DELAY STRING,
    ARR_DELAY STRING,
    CANCELLED STRING,
    DIVERTED STRING,
    DISTANCE STRING
  ",
  header => 'true'
)
LIMIT 10;


YEAR | MONTH | DAY_OF_MONTH | FL_DATE | UNIQUE_CARRIER | AIRLINE_ID | CARRIER | TAIL_NUM | FL_NUM | ORIGIN_AIRPORT_ID | ORIGIN_AIRPORT_SEQ_ID | ORIGIN | DEST_AIRPORT_ID | DEST_AIRPORT_SEQ_ID | DEST | DEP_DELAY | ARR_DELAY | CANCELLED | DIVERTED | DISTANCE
2016 | 7 | 2 | 2016-07-02 | DL | 19790 | DL | N956DL | 1367 | 11986 | 1198603 | GRR | 10397 | 1039705 | ATL | -2.00 | -16.00 | 0.00 | 0.00 | 640.00
2016 | 7 | 2 | 2016-07-02 | DL | 19790 | DL | N362NB | 1368 | 11278 | 1127803 | DCA | 13487 | 1348702 | MSP | 38.00 | 48.00 | 0.00 | 0.00 | 931.00
2016 | 7 | 2 | 2016-07-02 | DL | 19790 | DL | N991AT | 1369 | 10208 | 1020803 | AGS | 10397 | 1039705 | ATL | -4.00 | -18.00 | 0.00 | 0.00 | 143.00
2016 | 7 | 2 | 2016-07-02 | DL | 19790 | DL | N554NW | 1370 | 11433 | 1143302 | DTW | 14771 | 1477102 | SFO | 93.00 | 77.00 | 0.00 | 0.00 | 2079.00
2016 | 7 | 2 | 2016-07-02 | DL | 19790 | DL | N608AT | 1371 | 13485 | 1348502 | MSN | 10397 | 1039705 | ATL | 0.00 | -25.00 | 0.00 | 0.00 | 707.00
2016 |

7. Run a query on directly on the CSV files to return the top 20 routes by the number of flights that had taken place (a route is a unique origin-destination pair, we consider A->B and B->A as different routes). 

- Display `origin_airport_id`, `origin`, `dest_airport_id`, `dest`, `distance`, `num_flights`, `num_airlines_served` (you need to calculate the last two fields), where `num_airlines_servered` is a count of distinct airlines serving a route. 
- Order results by the descending order of `num_flights`. 


> Tip: to get the unique airlines served, consider `count(distinct ...)`

> Tip 2: In HiveQL summary queries, fields such as `origin` must be included in the group by fields in order to be shown in the final results.

In [ ]:
%sql
SELECT
  origin_airport_id,
  origin,
  dest_airport_id,
  dest,
  distance,
  COUNT(*) AS num_flights,
  COUNT(DISTINCT AIRLINE_ID) AS num_airlines_served
FROM read_files(
  "/Volumes/workspace/msbabigdata/flights/",
  format => "csv",
  schema => "YEAR STRING, MONTH STRING, DAY_OF_MONTH STRING, FL_DATE STRING, UNIQUE_CARRIER STRING, AIRLINE_ID STRING, CARRIER STRING, TAIL_NUM STRING, FL_NUM STRING, ORIGIN_AIRPORT_ID STRING, ORIGIN_AIRPORT_SEQ_ID STRING, ORIGIN STRING, DEST_AIRPORT_ID STRING, DEST_AIRPORT_SEQ_ID STRING, DEST STRING, DEP_DELAY STRING, ARR_DELAY STRING, CANCELLED STRING, DIVERTED STRING, DISTANCE STRING",
  header => "true"
)
GROUP BY
  ORIGIN_AIRPORT_ID, ORIGIN,
  DEST_AIRPORT_ID, DEST,
  DISTANCE
ORDER BY num_flights DESC
LIMIT 20;











origin_airport_id | origin | dest_airport_id | dest | distance | num_flights | num_airlines_served
14771 | SFO | 12892 | LAX | 337.00 | 17270 | 6
12892 | LAX | 14771 | SFO | 337.00 | 16894 | 6
12892 | LAX | 12478 | JFK | 2475.00 | 12481 | 4
12478 | JFK | 12892 | LAX | 2475.00 | 12476 | 4
12892 | LAX | 12889 | LAS | 236.00 | 11397 | 7
12889 | LAS | 12892 | LAX | 236.00 | 11381 | 7
14747 | SEA | 12892 | LAX | 954.00 | 10217 | 7
12892 | LAX | 14747 | SEA | 954.00 | 10208 | 7
12953 | LGA | 13930 | ORD | 733.00 | 9819 | 4
13930 | ORD | 12953 | LGA | 733.00 | 9730 | 4
12892 | LAX | 13930 | ORD | 1744.00 | 9171 | 5
13830 | OGG | 12173 | HNL | 100.00 | 9141 | 1
12173 | HNL | 13830 | OGG | 100.00 | 9061 | 1
12889 | LAS | 14771 | SFO | 414.00 | 8968 | 6
14771 | SFO | 12889 | LAS | 414.00 | 8959 | 6
13204 | MCO | 10397 | ATL | 404.00 | 8933 | 4
10397 | ATL | 13204 | MCO | 404.00 | 8910 | 4
13930 | ORD | 12892 | LAX | 1744.00 | 8818 | 5
10721 | BOS | 11278 | DCA | 399.00 | 8267 | 2
11278 | DCA | 1

In [ ]:
%sql
SELECT
  origin_airport_id,
  origin,
  dest_airport_id,
  dest,
  distance,
  COUNT(*) AS num_flights,
  COUNT(DISTINCT AIRLINE_ID) AS num_airlines_served
FROM read_files(
  "/Volumes/workspace/msbabigdata/flights/",
  format => "csv",
  schema => "YEAR STRING, MONTH STRING, DAY_OF_MONTH STRING, FL_DATE STRING, UNIQUE_CARRIER STRING, AIRLINE_ID STRING, CARRIER STRING, TAIL_NUM STRING, FL_NUM STRING, ORIGIN_AIRPORT_ID STRING, ORIGIN_AIRPORT_SEQ_ID STRING, ORIGIN STRING, DEST_AIRPORT_ID STRING, DEST_AIRPORT_SEQ_ID STRING, DEST STRING, DEP_DELAY STRING, ARR_DELAY STRING, CANCELLED STRING, DIVERTED STRING, DISTANCE STRING",
  header => "true"
)
WHERE CANCELLED = '0.00' OR CANCELLED IS NULL
GROUP BY
  ORIGIN_AIRPORT_ID, ORIGIN,
  DEST_AIRPORT_ID, 
  DEST,
  DISTANCE
ORDER BY num_flights DESC
LIMIT 20;

origin_airport_id | origin | dest_airport_id | dest | distance | num_flights | num_airlines_served
14771 | SFO | 12892 | LAX | 337.00 | 17270 | 6
12892 | LAX | 14771 | SFO | 337.00 | 16894 | 6
12892 | LAX | 12478 | JFK | 2475.00 | 12481 | 4
12478 | JFK | 12892 | LAX | 2475.00 | 12476 | 4
12892 | LAX | 12889 | LAS | 236.00 | 11397 | 7
12889 | LAS | 12892 | LAX | 236.00 | 11381 | 7
14747 | SEA | 12892 | LAX | 954.00 | 10217 | 7
12892 | LAX | 14747 | SEA | 954.00 | 10208 | 7
12953 | LGA | 13930 | ORD | 733.00 | 9819 | 4
13930 | ORD | 12953 | LGA | 733.00 | 9730 | 4
12892 | LAX | 13930 | ORD | 1744.00 | 9171 | 5
13830 | OGG | 12173 | HNL | 100.00 | 9141 | 1
12173 | HNL | 13830 | OGG | 100.00 | 9061 | 1
12889 | LAS | 14771 | SFO | 414.00 | 8968 | 6
14771 | SFO | 12889 | LAS | 414.00 | 8959 | 6
13204 | MCO | 10397 | ATL | 404.00 | 8933 | 4
10397 | ATL | 13204 | MCO | 404.00 | 8910 | 4
13930 | ORD | 12892 | LAX | 1744.00 | 8818 | 5
10721 | BOS | 11278 | DCA | 399.00 | 8267 | 2
11278 | DCA | 1

#### Create `flights` table and Query it

8. Now we can create a Managed table `flights.flights` (in the schema `flights`), using the same approach in Lab 3-1 (with Spark scripts). 

- Be sure to create the schema first, using SQL command `CREATE SCHEMA IF NOT EXISTS flights` or using Catalog UI.
- Make sure you use the right options for `spark.read` that match this dataset. Specifically, for this dataset, you should consider:

```
df = spark.read.option("header","true").option("delimiter",",")
```





In [ ]:


spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.flights")

# load flights CSV data
df = (
    spark.read
    .option("header", "true")
    .option("delimiter", ",")
    .schema("""
        YEAR STRING, MONTH STRING, DAY_OF_MONTH STRING, FL_DATE STRING,
        UNIQUE_CARRIER STRING, AIRLINE_ID STRING, CARRIER STRING, TAIL_NUM STRING,
        FL_NUM STRING, ORIGIN_AIRPORT_ID STRING, ORIGIN_AIRPORT_SEQ_ID STRING,
        ORIGIN STRING, DEST_AIRPORT_ID STRING, DEST_AIRPORT_SEQ_ID STRING,
        DEST STRING, DEP_DELAY STRING, ARR_DELAY STRING,
        CANCELLED STRING, DIVERTED STRING, DISTANCE STRING
    """)
    .csv("/Volumes/workspace/msbabigdata/flights/")
)

df.write.mode("overwrite").saveAsTable("workspace.flights.flights")







9. Query the first 10 rows of the newly created table. 

This time, you can directly use the `schema.table` (`flights.flights`) since the table has been created.

In [ ]:
%sql


SELECT *
FROM flights.flights
LIMIT 10;



YEAR | MONTH | DAY_OF_MONTH | FL_DATE | UNIQUE_CARRIER | AIRLINE_ID | CARRIER | TAIL_NUM | FL_NUM | ORIGIN_AIRPORT_ID | ORIGIN_AIRPORT_SEQ_ID | ORIGIN | DEST_AIRPORT_ID | DEST_AIRPORT_SEQ_ID | DEST | DEP_DELAY | ARR_DELAY | CANCELLED | DIVERTED | DISTANCE
2016 | 10 | 3 | 2016-10-03 | AA | 19805 | AA | N786AA | 1 | 10721 | 1072102 | BOS | 12478 | 1247803 | JFK | -4.00 | -6.00 | 0.00 | 0.00 | 187.00
2016 | 10 | 4 | 2016-10-04 | AA | 19805 | AA | N794AA | 1 | 10721 | 1072102 | BOS | 12478 | 1247803 | JFK | -6.00 | -3.00 | 0.00 | 0.00 | 187.00
2016 | 10 | 1 | 2016-10-01 | AA | 19805 | AA | N783AA | 1 | 12478 | 1247803 | JFK | 12892 | 1289204 | LAX | -7.00 | -30.00 | 0.00 | 0.00 | 2475.00
2016 | 10 | 2 | 2016-10-02 | AA | 19805 | AA | N798AA | 1 | 12478 | 1247803 | JFK | 12892 | 1289204 | LAX | 17.00 | 18.00 | 0.00 | 0.00 | 2475.00
2016 | 10 | 3 | 2016-10-03 | AA | 19805 | AA | N786AA | 1 | 12478 | 1247803 | JFK | 12892 | 1289204 | LAX | -5.00 | -10.00 | 0.00 | 0.00 | 2475.00
2016 | 10 | 4 

10. Replicate the query in step B7, but this time please use the newly created table.

Do you notice a query time difference? If so, this is because, when we create the new table, it is stored in a new optimaized Delta table format (a variation of Parquet), rather than CSV. Parquet format leads to smaller data files and faster query perforamnce compared to CSV.

In [ ]:
%sql
SELECT
  origin_airport_id,
  origin,
  dest_airport_id,
  dest,
  distance,
  COUNT(*) AS num_flights,
  COUNT(DISTINCT AIRLINE_ID) AS num_airlines_served
FROM flights.flights
WHERE CANCELLED = '0.00' OR CANCELLED IS NULL
GROUP BY
  ORIGIN_AIRPORT_ID, ORIGIN,
  DEST_AIRPORT_ID, DEST,
  DISTANCE
ORDER BY num_flights DESC
LIMIT 20;


origin_airport_id | origin | dest_airport_id | dest | distance | num_flights | num_airlines_served
14771 | SFO | 12892 | LAX | 337.00 | 17270 | 6
12892 | LAX | 14771 | SFO | 337.00 | 16894 | 6
12892 | LAX | 12478 | JFK | 2475.00 | 12481 | 4
12478 | JFK | 12892 | LAX | 2475.00 | 12476 | 4
12892 | LAX | 12889 | LAS | 236.00 | 11397 | 7
12889 | LAS | 12892 | LAX | 236.00 | 11381 | 7
14747 | SEA | 12892 | LAX | 954.00 | 10217 | 7
12892 | LAX | 14747 | SEA | 954.00 | 10208 | 7
12953 | LGA | 13930 | ORD | 733.00 | 9819 | 4
13930 | ORD | 12953 | LGA | 733.00 | 9730 | 4
12892 | LAX | 13930 | ORD | 1744.00 | 9171 | 5
13830 | OGG | 12173 | HNL | 100.00 | 9141 | 1
12173 | HNL | 13830 | OGG | 100.00 | 9061 | 1
12889 | LAS | 14771 | SFO | 414.00 | 8968 | 6
14771 | SFO | 12889 | LAS | 414.00 | 8959 | 6
13204 | MCO | 10397 | ATL | 404.00 | 8933 | 4
10397 | ATL | 13204 | MCO | 404.00 | 8910 | 4
13930 | ORD | 12892 | LAX | 1744.00 | 8818 | 5
10721 | BOS | 11278 | DCA | 399.00 | 8267 | 2
11278 | DCA | 1


11\. As a part of cleaning up, please remove all volumes/schema/tables that we have created in this homework.

- `DROP VOLUME IF EXISTS workspace.<schema>.<volume>`: drop a volume (including files in it)
- `DROP SCHEMA IF EXISTS workspace.<schema> CASCADE`: drop a database|schema, including tables in it. 

Verify using Catalog UI that both the flights volume under `msbabigdata` and the `flights` schema under workspace have been removed. 


In [ ]:
%sql
DROP VOLUME IF EXISTS workspace.msbabigdata.flights;
DROP SCHEMA IF EXISTS workspace.flights CASCADE;



## Submission Checklist:
- Please include everything in one file (HTML, PDF, or word). If you use PDF, please make sure there are no cutoffs of long lines.